In [ ]:
from wti_env import WTIEnv
import pandas as pd
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import random
import math

In [5]:
df = pd.read_csv("wti_1_data.csv")

In [9]:
feature_cols = [col for col in df.columns if col not in ["Date", "ret_CL1"]]
return_col = "ret_CL1"

In [ ]:
env = WTIEnv(
    df=df,
    feature_cols=feature_cols,
    return_col=return_col,
    window=20,
    cost=0.0005
)

obs = env.reset()
obs.shape

(20, 44)

In [12]:
obs = env.reset()
total_reward = 0

for _ in range(200):
    action = env.action_space.sample()  # random short/flat/long
    obs, reward, done, info = env.step(action)
    total_reward += reward
    if done:
        break

total_reward


-0.412663496454999

Train a DQN agent bc state space is huge

In [ ]:
state_dim = obs.size  # flatten the window
n_actions = env.action_space.n

class QNet(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions)
        
        
    def forward(self, x):
        return self.net(x)
        
qnet = QNet(state_dim, n_actions)
optimizer = optim.Adam(qnet.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()


In [18]:
from collections import deque

buffer = deque(maxlen=50000)

def store_transition(state, action, reward, next_state, done):
    buffer.append((state, action, reward, next_state, done))

def sample_batch(batch_size=32):
    batch = random.sample(buffer, batch_size)
    states, actions, rewards, next_states, dones = zip(*batch)
    return (
        torch.tensor(states, dtype=torch.float32),
        torch.tensor(actions, dtype=torch.long),
        torch.tensor(rewards, dtype=torch.float32),
        torch.tensor(next_states, dtype=torch.float32),
        torch.tensor(dones, dtype=torch.float32),
    )


In [19]:
def select_action(qnet, state, epsilon):
    if random.random() < epsilon:
        return random.randint(0, n_actions - 1)
    else:
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        q_values = qnet(state_t)
        return q_values.argmax().item()

In [30]:
episodes = 5
batch_size = 32
gamma = 0.99

epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.05

for ep in range(episodes):
    state = env.reset().flatten()
    total_reward = 0
    
    done = False
    while not done:
        
        action = select_action(qnet, state, epsilon)
        next_obs, reward, done, _ = env.step(action)
        next_state = next_obs.flatten()
        
        store_transition(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

        if math.isnan(reward):
            print("NAN reward")

        if math.isnan(total_reward):
            print("NAN total")
        
        # Update Q-network if enough samples:
        if len(buffer) > batch_size:
            states, actions, rewards, next_states, dones = sample_batch(batch_size)

            # Compute Q(s,a)
            q_values = qnet(states).gather(1, actions.unsqueeze(1)).squeeze()

            # Compute max_a' Q(s', a')
            next_q_values = qnet(next_states).max(dim=1)[0]

            targets = rewards + gamma * next_q_values * (1 - dones)

            loss = loss_fn(q_values, targets.detach())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    print(f"total_reward {total_reward}")
    # Decay exploration
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    print(f"Episode {ep}, reward = {total_reward:.4f}, epsilon={epsilon:.3f}")


NAN reward
NAN total
NAN reward
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN total
NAN tota

KeyboardInterrupt: 

In [22]:
state = env.reset().flatten()
actions = []
rewards = []

done = False
while not done:
    action = select_action(qnet, state, epsilon=0.0)  # deterministic
    actions.append(action)
    
    next_obs, reward, done, _ = env.step(action)
    state = next_obs.flatten()
    rewards.append(reward)

sum(rewards), actions[:20]

(nan, [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])